In [ ]:
# ============================================================
#  Configuration
# ============================================================
RESULTS_DIR = "results"
RUN_NAME    = "20260407_185821_H_G_GeForce_RTX_3090_x1_Qwen2.5-1.5B"
# ============================================================

import json, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 11})

run_dir = Path(RESULTS_DIR) / RUN_NAME
assert run_dir.exists(), f"Not found: {run_dir}"

def load_json(name):
    p = run_dir / name
    if p.exists():
        with open(p) as f: return json.load(f)
    return None

def load_csv(name):
    p = run_dir / name
    if p.exists():
        df = pd.read_csv(p)
        if 'elapsed_s' in df.columns:
            df['elapsed_s'] = df['elapsed_s'] - df['elapsed_s'].iloc[0]
        return df
    return None

# Auto-detect mode
if (run_dir / 'hybrid.json').exists():
    MODE = 'hybrid'
elif (run_dir / 'gpu_only.json').exists():
    MODE = 'gpu_only'
else:
    raise FileNotFoundError('No benchmark JSON found')

bench     = load_json(f'{MODE}.json')
sysinfo   = load_json('system_info.json') or {}
gpu_csv   = load_csv(f'{MODE}_monitor_gpu.csv')
cpu_csv   = load_csv(f'{MODE}_monitor_cpu.csv')

print(f'Run: {RUN_NAME}')
print(f'Mode: {MODE}')
print(f'Bench: {"YES" if bench else "NO"}')
print(f'GPU CSV: {len(gpu_csv)} rows' if gpu_csv is not None else 'GPU CSV: NO')
print(f'CPU CSV: {len(cpu_csv)} rows' if cpu_csv is not None else 'CPU CSV: NO')

In [ ]:
# ============================================================
#  Section 0: System & Config Info
# ============================================================

tbl_style = [
    {"selector":"th","props":[("background-color","#2d4a7a"),("color","white"),
                               ("font-size","12px"),("padding","5px 10px")]},
    {"selector":"tr:nth-child(even)","props":[("background-color","#f5f8ff")]},
    {"selector":"td","props":[("font-size","12px"),("padding","4px 10px")]},
]

# Hardware
cpu  = sysinfo.get('cpu', {})
gpu  = sysinfo.get('gpu', {})
mem  = sysinfo.get('memory', {})
sw   = sysinfo.get('software', {})

hw_rows = [
    ('Hostname',     sysinfo.get('hostname', '-')),
    ('CPU',          cpu.get('model_name', '-')),
    ('CPU Topology', f"{cpu.get('sockets','-')}S x {cpu.get('cores_per_socket','-')}C x {cpu.get('threads_per_core','-')}T = {cpu.get('total_cpus','-')} CPUs"),
    ('ISA Flags',    ', '.join(cpu.get('flags', []))),
    ('RAM',          f"{int(mem.get('total','0 kB').replace(' kB','').strip()) / 1024**2:.1f} GB" if mem.get('total') else '-'),
    ('GPU Count',    str(gpu.get('count', '-'))),
]
for dev in gpu.get('devices', []):
    hw_rows.append((f"GPU {dev['index']}", f"{dev.get('name','-')} ({int(dev.get('memory_total_mib',0))/1024:.0f}GB)"))
hw_rows += [
    ('PyTorch',  sw.get('torch_version', '-')),
    ('vLLM',     sw.get('vllm_version', '-')),
]

display(pd.DataFrame(hw_rows, columns=['Item','Value']).set_index('Item')
        .style.set_table_styles(tbl_style).set_caption('<b>Hardware & Software</b>'))

# Hybrid config
hc = sysinfo.get('hybrid_config', {})
if hc:
    strategy = hc.get('routing_strategy', '?')
    priority = hc.get('routing_priority', '?')
    routing_label = strategy if strategy == 'round-robin' else f'{strategy} ({priority})'
    hc_rows = [
        ('Routing',    routing_label),
        ('CPU Max Seqs', hc.get('cpu_max_seqs', '0')),
        ('CPU KVCache GB', hc.get('cpu_kvcache_gb', '0')),
        ('CPU Threads', hc.get('cpu_threads', '0')),
        ('NUMA Aware',  hc.get('numa_aware', '-')),
        ('CPU Engines', hc.get('num_cpu_engines', '1')),
    ]
    display(pd.DataFrame(hc_rows, columns=['Item','Value']).set_index('Item')
            .style.set_table_styles(tbl_style).set_caption(f'<b>Hybrid Config — {routing_label}</b>'))
elif MODE == 'gpu_only':
    print('GPU-only mode (no hybrid config)')

In [ ]:
# ============================================================
#  Section 1: Benchmark Results
# ============================================================

if bench:
    b_rows = [
        ('Model',         bench.get('model_id', '-')),
        ('Prompts',       f"{bench.get('completed','?')}/{bench.get('num_prompts','?')}"),
        ('Duration',      f"{bench.get('duration',0):.1f}s"),
        ('Wall Time',     f"{bench.get('wall_time_s',0):.1f}s" if bench.get('wall_time_s') else 'N/A'),
        ('Req TP',        f"{bench.get('request_throughput',0):.2f} req/s"),
        ('Output TP',     f"{bench.get('output_throughput',0):.0f} tok/s"),
        ('Total TP',      f"{bench.get('total_token_throughput',0):.0f} tok/s"),
        ('Mean TTFT',     f"{bench.get('mean_ttft_ms',0):.1f} ms"),
        ('Median TTFT',   f"{bench.get('median_ttft_ms',0):.1f} ms"),
        ('P99 TTFT',      f"{bench.get('p99_ttft_ms',0):.1f} ms"),
        ('Mean TPOT',     f"{bench.get('mean_tpot_ms',0):.2f} ms"),
        ('Median TPOT',   f"{bench.get('median_tpot_ms',0):.2f} ms"),
        ('P99 TPOT',      f"{bench.get('p99_tpot_ms',0):.2f} ms"),
        ('Mean ITL',      f"{bench.get('mean_itl_ms',0):.2f} ms"),
        ('P99 ITL',       f"{bench.get('p99_itl_ms',0):.2f} ms"),
    ]
    display(pd.DataFrame(b_rows, columns=['Metric','Value']).set_index('Metric')
            .style.set_table_styles(tbl_style).set_caption(f'<b>Benchmark — {MODE}</b>'))

In [ ]:
# ============================================================
#  Section 2: GPU Utilization Timeline
# ============================================================

if gpu_csv is not None and 'elapsed_s' in gpu_csv.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # GPU util
    ax = axes[0]
    gpu_util_cols = [c for c in gpu_csv.columns if c.startswith('gpu') and c.endswith('_util_pct') and 'avg' not in c]
    for c in gpu_util_cols:
        ax.plot(gpu_csv['elapsed_s'], gpu_csv[c], label=c.replace('_util_pct',''), alpha=0.8)
    if 'gpu_avg_util_pct' in gpu_csv.columns:
        ax.plot(gpu_csv['elapsed_s'], gpu_csv['gpu_avg_util_pct'], 'k--', lw=2, label='avg')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Utilization %')
    ax.set_title('GPU Utilization'); ax.legend(fontsize=9); ax.set_ylim(-5, 105)
    ax.grid(True, alpha=0.3)

    # GPU power
    ax = axes[1]
    power_cols = [c for c in gpu_csv.columns if c.endswith('_power_w') and 'avg' not in c]
    for c in power_cols:
        ax.plot(gpu_csv['elapsed_s'], gpu_csv[c], label=c.replace('_power_w',''), alpha=0.8)
    if 'gpu_avg_power_w' in gpu_csv.columns:
        ax.plot(gpu_csv['elapsed_s'], gpu_csv['gpu_avg_power_w'], 'k--', lw=2, label='avg')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Power (W)')
    ax.set_title('GPU Power'); ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('No GPU monitor data')

In [ ]:
# ============================================================
#  Section 3: CPU Utilization Timeline
# ============================================================

if cpu_csv is not None and 'elapsed_s' in cpu_csv.columns:
    core_cols = [c for c in cpu_csv.columns if c.startswith('core') and c.endswith('_util_pct')]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Average CPU util
    ax = axes[0]
    if 'cpu_avg_util_pct' in cpu_csv.columns:
        ax.plot(cpu_csv['elapsed_s'], cpu_csv['cpu_avg_util_pct'], 'b-', lw=2, label='avg')
        ax.fill_between(cpu_csv['elapsed_s'], cpu_csv['cpu_avg_util_pct'], alpha=0.2)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Utilization %')
    ax.set_title('CPU Average Utilization'); ax.set_ylim(-5, 105)
    ax.legend(); ax.grid(True, alpha=0.3)

    # Per-core heatmap
    ax = axes[1]
    if core_cols:
        core_data = cpu_csv[core_cols].values.T
        im = ax.imshow(core_data, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100,
                       extent=[cpu_csv['elapsed_s'].iloc[0], cpu_csv['elapsed_s'].iloc[-1],
                               len(core_cols)-0.5, -0.5])
        ax.set_xlabel('Time (s)'); ax.set_ylabel('Core')
        ax.set_title('Per-Core Utilization Heatmap')
        plt.colorbar(im, ax=ax, label='%')

    plt.tight_layout()
    plt.show()
else:
    print('No CPU monitor data')

In [ ]:
# ============================================================
#  Section 4: GPU Memory Utilization
# ============================================================

if gpu_csv is not None and 'elapsed_s' in gpu_csv.columns:
    mem_cols = [c for c in gpu_csv.columns if c.endswith('_mem_util_pct') and 'avg' not in c]
    if mem_cols:
        fig, ax = plt.subplots(figsize=(14, 4))
        for c in mem_cols:
            ax.plot(gpu_csv['elapsed_s'], gpu_csv[c], label=c.replace('_mem_util_pct',''), alpha=0.8)
        ax.set_xlabel('Time (s)'); ax.set_ylabel('Memory %')
        ax.set_title('GPU Memory Utilization'); ax.legend(fontsize=9)
        ax.set_ylim(-5, 105); ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()
    else:
        print('No GPU memory columns')
else:
    print('No GPU monitor data')